In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, udf, to_timestamp, sum
from pyspark.sql.types import *
import os
from pathlib import Path
 
# Set HADOOP_HOME and update PATH
hadoop_home = "C:\\vs_code_projects\\python\\hadoop-3.2.1"
os.environ['HADOOP_HOME'] = str(hadoop_home)
os.environ['PATH'] += os.pathsep + str(hadoop_home + '\\bin')

# Create Spark session
print('starting session')
spark = SparkSession. \
        builder. \
        appName("NYC Taxi Pipeline"). \
        config("spark.executor.cores", 1). \
        config("spark.executor.instances", 1). \
        config("spark.executor.memory", "1g"). \
        config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"). \
        config("spark.hadoop.io.native.lib.available", "false"). \
        master("local[*]"). \
        getOrCreate()
print('session created')

starting session
session created


In [3]:
taxi01_df = spark.read.parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\01_taxi01")
taxi02_df = spark.read.parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\01_taxi02")
zone_df = spark.read.parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\01_zone")


In [7]:
def clean_taxi_df(df):
    # Remove invalid trips
    df = df.filter(
        (col("trip_distance") >= 0) &
        (col("fare_amount") >= 0)
    )
    
    # Handle nulls in important columns
    df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "PULocationID", "DOLocationID", "payment_type", "fare_amount", "total_amount"])
    
    return df


In [8]:
taxi01_clean_df = clean_taxi_df(taxi01_df)
print(f"Rows dropped for taxi01_df: {taxi01_df.count() - taxi01_clean_df.count()}")


Rows dropped for taxi01_df: 4542


In [10]:
taxi02_clean_df = clean_taxi_df(taxi02_df)
print(f"Rows dropped for taxi02_df: {taxi02_df.count() - taxi02_clean_df.count()}")


Rows dropped for taxi02_df: 4307


In [34]:
zone_df.show(4)

+----------+---------+--------------------+------------+
|LocationID|  Borough|                Zone|service_zone|
+----------+---------+--------------------+------------+
|         1|      EWR|      Newark Airport|         EWR|
|         2|   Queens|         Jamaica Bay|   Boro Zone|
|         3|    Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|Manhattan|       Alphabet City| Yellow Zone|
+----------+---------+--------------------+------------+
only showing top 4 rows


In [13]:
zone_clean_df = zone_df.dropna(subset=["LocationID", "Borough", "Zone", "service_zone"])

print(f"Rows dropped for zone_df: {zone_df.count() - zone_clean_df.count()}")


Rows dropped for zone_df: 0


In [14]:
taxi01_clean_df.write.mode("overwrite").parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\taxi01")

In [15]:
taxi02_clean_df.write.mode("overwrite").parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\taxi02")

In [16]:
zone_clean_df.write.mode("overwrite").parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\02_clean_data\\zone")

In [17]:
spark.stop()